In [ ]:
import os
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
import gradio as gr

load_dotenv()

### Initialize Embeddings Vector store and Retriever

In [ ]:
print("Initializing components...")

embeddings = OpenAIEmbeddings(  model="text-embedding-3-small",openai_api_key=os.environ.get("OPENAI_API_KEY"))
llm = ChatOpenAI(model="gpt-4o-mini",temperature=0.2, max_tokens=2000)

vectorstore = PineconeVectorStore(
    index_name=os.environ["INDEX_NAME"], embedding=embeddings
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [ ]:
# Query
# query = "who is Chief Executive Officer of ACME CORPORATION?"
# query = "what is return and refund policy?"
# query = "who is Margaret Ellison?"
query = "who is the CEO of ACME CORPORATION?"

## Invoke without RAG

In [ ]:
result_raw = llm.invoke([HumanMessage(content=query)])
print("\nAnswer:")
print(result_raw.content)

In [ ]:
def format_docs(docs):
    """Format retrieved documents into a single string."""
    return "\n\n".join(doc.page_content for doc in docs)

## Step 1: Retrieve relevant documents

In [ ]:

docs = retriever.invoke(query)

context = format_docs(docs)    
   
print("\nRetrieved Context:")
for i, doc in enumerate(docs):
    print(f"\nDocument {i+1}:\n{doc.page_content}\n")


## Step 2: Format the prompt with context and question

In [ ]:
prompt_template = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:

{context}

Question: {question}

Provide a detailed answer:"""
)


messages = prompt_template.format_messages(context=context, question=query)

print("\nFormatted Prompt:")
for message in messages:
    print(f"{message.type}: {message.content}")

## Step 3: Invoke LLM with the formatted messages

In [ ]:

response = llm.invoke(messages) 
print (response.content)

## Make it a fuction

In [ ]:
def answer_query(query: str,history):
    docs = retriever.invoke(query)
    context = format_docs(docs)    
    messages = prompt_template.format_messages(context=context, question=query)
    response = llm.invoke(messages)
    return response.content

## Bring this up in Gradio using the Chat interface 

In [ ]:
gr.ChatInterface(answer_query).launch()